# Current proposals and one fixed corner-suppression experiment

**CPU only. Run all four code cells. No optimizer or training is used.**
The first analysis runs fresh inference with the frozen PR158 reviewed-label
checkpoint on the same 20 development photographs, in both original and
reflected views. It captures every decoded FP32 box, score and corner before
confidence filtering and native suppression, then verifies the exact native
reconstruction and records candidate lineage.

The second analysis uses the previously saved R68 outputs for five methods.
It compares unchanged outputs, additional box suppression at IoU 0.60, and
the same fixed rule qualified by two accepted corresponding corners within
five pixels. It has a separate protocol and produces separate evidence.
The qualified rule is also checked on the fresh current-checkpoint outputs.

Both analyses retain r50, r67, supplied664 and frozen666 references, with r67
primary. Thresholds, labels and model weights are unchanged. Selection uses
predictions alone; reference-based candidate inspection is diagnostic.
Results describe development evidence and do not automatically promote a
model or postprocessing rule.

Inputs include one approximately 21.7 MB checkpoint, 14.9 MB of original
photographs and about 4.3 MB of archived evidence. The raw return preserves
about 34 MB of FP32 tensor values using lossless compression.


## 1 Load the published source and both fixed protocols

Source, cached inputs and outputs remain separate. An edited source checkout
is preserved and rejected; choose another `BAOMAI_CURRENT_PROPOSALS_BASE`
to run beside it. Every project module and both protocols are hash-checked.


In [1]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

SOURCE_COMMIT = "41d0e6937c695377373cc55071ffb780345870ee"
SOURCE_HASHES = {'diagnose_current_proposals_r69.py': '5e6e6a89e89e5f47adf513fa47dc79028895d0a06133cdbf685850cc1cec4b3b', 'analyze_partial_detection_r68.py': '18a2f0f35d051d87fb158c3a0908e461cf85c420b8f02b59ebb42574984f2d5a', 'analyze_duplicate_audit_r66.py': '868ab2c5df33ba376da4212aca6281d99a9d5c3a9d6cd1dac7dfaf4e1cd7acfa', 'analyze_proposal_stages_r63.py': '0dd585cb7d957dcd599c0f40fbbf8fcc1ffc85c9bf98f98307eb0c799f29f15b', 'audit_additional_front_pilot_r45_results.py': '4f2d5ddf167ae480ede653fa821efd8b36c5679aed9d3e79e98b1d7f71f4de3b', 'audit_additional_front_pilot_r47_results.py': '6d7a2d3ece42b9eb26a473437a8f8377474dee25d4b83c14a80cb04ab03d64d9', 'audit_front_diagnosis_r49_results.py': '5bb583f1cd9adfbf1cbef5084df935a6c2b2587f5b6c730b49ea26029d1244e2', 'audit_reflection_return.py': '55f391535b040a27682e899aec74915b87864063023bc1ed818f8f82e67101de', 'diagnose_fronts_r50.py': 'cba4f1df6dcf431c933af1506a4de15718636d9477406eb34779ac4d8ab6ee3e', 'evaluate_pose.py': '3f80bd09efbe97327b705ffe6ad3e662111f3e2bc71b69b51a9d38d7baa90ce0', 'export_front_error_review_r48.py': '45f8e98b2ecd77fc5cdd2db8debb018dfa3904cd373b875049eb491eb62227b5', 'pose_checkpoint.py': 'c1311307fd49b37e60f2939c9b084bc36690c1d13b6b31b9e4c51ea9353562d4', 'pose_scoring.py': '04e5cdbef26df978707d0b6e4b82f05471b6012845f7986b9c05c71644ec0018', 'predict_pose_folder.py': '2942864e86303a67911826fc0dbfe547b4b4fc2ed3860e74c9d0210c635520a6', 'replay_front_confidence_r51.py': '97828d8423f6b4f693a36c42bc23d1b868540bdfa6b75bfaf42373cffc5ba8c0', 'replay_partial_inclusion_r67.py': '5502022e8f5a7b510e875f2948d3dcbf4b1ab59addd94a9e5d3260ec3eda7f9a', 'run_corner_average_r58.py': '286232cf490f950b367068ea52b43078970ab27bb016169cb99159dab2fdbd88', 'run_flip_inference_r56.py': 'bf2ff9056f12976ca8c4aa971de2d2a7af1c679369c44dbf73de9f9d2542700d', 'run_inference_ablation_r53.py': '8c8a37c3dcaccde5c568e9df6f44651cdb05965ca5f5ed5b0f2978852387dcc1', 'run_original_comparison_r59.py': '279e2c0bd701299ea2a6f30403e97da68cc44a9c4ff874c5ee5a0d155a6e6ea3', 'run_reference_matrix_r60.py': '2939b2789abf2566d7680de60a0a87efae2a41e45c2dc01c9d15d1221cb8bbf1', 'run_training_label_comparison_r61.py': '0dc8d5dfdd344f590f88d17ea7ced76f3fc018026fae221a4df6e25d7c807a9c', 'analyze_corner_suppression_r69.py': '719ef2406a2d4a95311ff6ad866e161558a1b8e1b6632b3f60bd78b0c29e291f'}
PROTOCOL_HASHES = {'docs/thesis/evidence/current-proposals-r69/protocol.json': '86a1808314962c2ced91ea6d0af757dac6adb6ff05b02149dae3f6bd0a3108f8', 'docs/thesis/evidence/corner-suppression-r69/protocol.json': '27eebb3c8e1c993e849ca3c9acf7be39d1ca38c9dd3472ff75003d74672af1e6'}
BASE = Path(os.environ.get("BAOMAI_CURRENT_PROPOSALS_BASE", "/content/baomai-current-proposals-r69")).resolve()
SOURCE, WORK = BASE / "source", BASE / "work"
INPUT_DIR, OUTPUT, COMPANION = WORK / "inputs", WORK / "current-proposals", WORK / "corner-suppression"
BASE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0", "PYTHONDONTWRITEBYTECODE": "1"}
def run(command, cwd=None):
    subprocess.run(list(map(str, command)), cwd=cwd, env=ENV, check=True)
def git(*args):
    return subprocess.check_output(["git", *args], cwd=SOURCE, env=ENV, text=True).strip()
if not (SOURCE / ".git").exists():
    if SOURCE.exists() and any(SOURCE.iterdir()):
        raise RuntimeError("Existing files preserved; choose a fresh BAOMAI_CURRENT_PROPOSALS_BASE.")
    run(["git", "init", SOURCE])
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], SOURCE)
    run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_COMMIT], SOURCE)
    run(["git", "sparse-checkout", "init", "--no-cone"], SOURCE)
    run(["git", "sparse-checkout", "set", "/scripts/", "/docs/thesis/evidence/partial-detection-r68/baomai-partial-detection-r68-results.zip",
         "/data/reviews/fronts-2026-11-28-r67/", "/docs/thesis/evidence/partial-inclusion-r67/excluded-front-provenance.json",
         "/docs/thesis/evidence/current-proposals-r69/protocol.json", "/docs/thesis/evidence/corner-suppression-r69/protocol.json"], SOURCE)
    run(["git", "checkout", "--detach", SOURCE_COMMIT], SOURCE)
def verify_source():
    if git("rev-parse", "HEAD") != SOURCE_COMMIT or git("status", "--porcelain", "--untracked-files=all"):
        raise RuntimeError("Source changed; choose a fresh BAOMAI_CURRENT_PROPOSALS_BASE.")
    for name, expected in SOURCE_HASHES.items():
        if hashlib.sha256((SOURCE / "scripts" / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError("Implementation hash differs: " + name)
    for name, expected in PROTOCOL_HASHES.items():
        if hashlib.sha256((SOURCE / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError("Fixed protocol changed: " + name)
verify_source()
WORK.mkdir(parents=True, exist_ok=True)
os.environ["YOLO_CONFIG_DIR"] = str(WORK / "runtime-config")
ENV["YOLO_CONFIG_DIR"] = os.environ["YOLO_CONFIG_DIR"]
print("Immutable source:", SOURCE_COMMIT)


Immutable source: 41d0e6937c695377373cc55071ffb780345870ee


## 2 Verify the runtime, checkpoint, photographs and references

Existing Torch and torchvision installations are retained. CPU packages are
installed only if they are absent. The pinned Ultralytics, NumPy, OpenCV and
Pillow versions are installed only when missing or different. Critical
native source files are checked before inference; all actual versions are
recorded, including CPU versus the archived L4 environment.

Reuse verified input files when present. Otherwise obtain the checkpoint
parts and photographs from immutable GitHub paths, and verify complete
hashes. No training label or reference file is written.


In [2]:
from importlib.util import find_spec
from importlib.metadata import version, PackageNotFoundError
if find_spec("torch") is None or find_spec("torchvision") is None:
    run([sys.executable, "-m", "pip", "install", "--quiet", "--index-url",
         "https://download.pytorch.org/whl/cpu", "torch==2.10.0+cpu", "torchvision==0.25.0+cpu"])
required = {"ultralytics": "8.4.182", "numpy": "2.4.6", "opencv-python": "5.0.0.93", "Pillow": "13.0.0"}
def installed(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None
missing = [name + "==" + expected for name, expected in required.items() if installed(name) != expected]
if missing:
    constraints = WORK / "torch-constraints.txt"
    constraints.write_text("".join(name + "==" + version(name) + "\n" for name in ("torch", "torchvision")))
    run([sys.executable, "-m", "pip", "install", "--quiet", "--constraint", constraints, *missing])
sys.dont_write_bytecode = True
sys.path.insert(0, str(SOURCE / "scripts"))
import diagnose_current_proposals_r69 as audit
plan = audit.prepare(SOURCE, INPUT_DIR)
runtime = audit.native_runtime()
print("Frozen checkpoint:", plan["checkpoint"])
print("Verified photographs:", len(plan["images"]))
print("Runtime packages:", runtime["packages"])


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/workspace/scratch/be67d3adbd23/r69-notebook-run/work/runtime-config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Frozen checkpoint: {'filename': 'reviewed_labels-last.pt', 'bytes': 21704814, 'sha256': 'c749fcff2126467cbc770f75a71fd79da51ebd08308ba909a448c3dd8cb5b372'}
Verified photographs: 20
Runtime packages: {'torch': '2.10.0+cpu', 'torchvision': '0.25.0+cpu', 'ultralytics': '8.4.182', 'numpy': '2.4.6', 'opencv-python': '5.0.0.93', 'Pillow': '13.0.0'}


## 3 Collect all proposals and run the two bounded analyses

Capture both views at the unchanged confidence/NMS operating point. Preserve
raw FP32 tensors and exact coordinate conversion, native returned indices,
rank/tie witnesses and postclip lineage. The frozen model state must remain
unchanged after normal inference warmup. Compare fresh CPU outputs with the
archived L4 results while retaining numerical and scored-outcome differences.

Diagnose first-lost stages, view support, fixed-pool coverage and duplicate
competitions. Candidate box quality and confidence-qualified known-corner
errors remain separate. Then run the companion's fixed saved-output rule and
verify its reviewed-checkpoint result against the fresh-runtime rule check.


In [3]:
summary, records = audit.analyze(SOURCE, INPUT_DIR, OUTPUT, plan)
audit.write_outputs(SOURCE, OUTPUT, summary, records)
run([sys.executable, "-B", SOURCE / "scripts/analyze_corner_suppression_r69.py",
     "--r68-result-zip", INPUT_DIR / audit.INPUTS["r68"]["filename"], "--output", COMPANION])
audit.verify_companion_continuity(records["qualified-rule-runtime-continuity"], COMPANION)
from IPython.display import Markdown, display
display(Markdown((OUTPUT / "results.md").read_text()))
display(Markdown((COMPANION / "results.md").read_text()))
print("Raw diagnostic records:", summary["record_counts"])
print("Captured raw archive:", summary["decoded_archive"])


Raw diagnostic records: {'forward_views': 40, 'per_target_view_stages': 5284, 'stage_coverage_graphs': 640, 'duplicate_output_records': 294, 'cross_runtime_discrete_target_changes': 666, 'cross_runtime_record_comparisons': 100, 'cross_runtime_duplicate_boundary_records': 6, 'per_image_score_objects': 800, 'combined_view_reference_records': 2642}
Captured raw archive: {'filename': 'decoded-proposals.npz', 'bytes': 29296868, 'sha256': 'c26ccba90d28a6dab9c2f6bd1049b19374d0ddbf60594377ab9e81332454e2b6', 'arrays': 40, 'uncompressed_tensor_bytes': 34003200, 'quantization': 'none; FP32 preserved'}


# Current reviewed-checkpoint proposal stages

Fresh CPU inference collected every decoded FP32 proposal from the frozen PR158 reviewed-label checkpoint for all 20 development photographs in both original and reflected views. The confidence, NMS, matching and corner thresholds remain unchanged. No optimizer, training or label editing is used.

## r67 measurements and runtime comparison

| Runtime | Output stage | Misses | Extras | Duplicate subset | Pair ≤5 px | Pair ≤10 px |
|---|---|---:|---:|---:|---:|---:|
| archived_l4 | native_original | 18 | 57 | 19 | 570/583 | 582/583 |
| archived_l4 | native_reflected | 12 | 47 | 17 | 568/583 | 583/583 |
| archived_l4 | postclip_original | 18 | 50 | 13 | 570/583 | 582/583 |
| archived_l4 | postclip_reflected | 12 | 39 | 9 | 568/583 | 583/583 |
| archived_l4 | two_view | 18 | 50 | 13 | 572/583 | 583/583 |
| fresh_cpu | native_original | 18 | 57 | 20 | 570/583 | 582/583 |
| fresh_cpu | native_reflected | 12 | 47 | 17 | 568/583 | 583/583 |
| fresh_cpu | postclip_original | 18 | 50 | 14 | 570/583 | 582/583 |
| fresh_cpu | postclip_reflected | 12 | 39 | 9 | 568/583 | 583/583 |
| fresh_cpu | two_view | 18 | 50 | 14 | 572/583 | 583/583 |

The archive/fresh comparison records 666 target-level index/outcome differences across five output stages and four references. Of these, 666 are matched-output index changes with unchanged detection, known-corner availability and 5/10-pixel statuses. Detection gains/losses are 0/0; complete-pair five/ten-pixel status changes are 0/0. Full numerical differences and per-image scores are retained. A changed record index does not by itself establish a changed physical match.

The duplicate-category comparison is separate. Its ledger preserves any unmatched-output reference-IoU boundary crossing and the corresponding fresh native suppression witness. A closest fresh raw box to an archived output is explicitly a diagnostic association; archived full decoded anchors remain unavailable.

## Fresh r67 first-lost stages

| View | Stage | Fronts |
|---|---|---:|
| original | confidence_gate | 12 |
| original | matched | 640 |
| original | native_nms | 1 |
| original | no_iou_eligible_decoded_box | 5 |
| reflected | confidence_gate | 10 |
| reflected | matched | 646 |
| reflected | native_nms | 1 |
| reflected | no_iou_eligible_decoded_box | 1 |

## Three restored focus targets

| Target | View | Stage | Best raw IoU | Highest IoU-eligible score | Eligible raw boxes |
|---|---|---|---:|---:|---:|
| r67/new_000009/R055 | original | confidence_gate | 0.595507 | 0.01454296 | 2 |
| r67/new_000009/R056 | original | no_iou_eligible_decoded_box | 0.446675 | None | 0 |
| r67/new_000053/R022 | original | confidence_gate | 0.589806 | 0.00016235 | 3 |
| r67/new_000009/R055 | reflected | confidence_gate | 0.690462 | 0.00772521 | 3 |
| r67/new_000009/R056 | reflected | confidence_gate | 0.525817 | 0.08451592 | 1 |
| r67/new_000053/R022 | reflected | confidence_gate | 0.749237 | 0.00014884 | 2 |

The best-overlap and highest-scoring eligible raw boxes are separate records. Each includes exact geometry, candidate identity, stage survival and confidence-qualified errors at known corners. A geometrically eligible proposal does not automatically supply a useful corner estimate.

The duplicate-competition ledger compares each scored duplicate with the matched outputs sharing its eligible reference: their mutual box overlap, confidence order, individual reference overlaps and known-corner errors remain visible. Matching IoU 0.50 to the same reference does not imply that their mutual overlap exceeds native NMS 0.70.

## Fixed corner-rule runtime continuity

The separately frozen corner-qualified suppression rule is also applied unchanged to this checkpoint's fresh completed two-view outputs. This is a runtime continuity check, not a new threshold comparison.

| Runtime | r67 misses | Extras | Duplicate subset | Pair ≤5 px | Pair ≤10 px |
|---|---:|---:|---:|---:|---:|
| archived_l4 | 18 | 44 | 11 | 572/583 | 583/583 |
| fresh_cpu | 18 | 44 | 12 | 572/583 | 583/583 |

All-image selected-index agreement: True. Cross-runtime discrete target changes after the rule: 116. Fresh baseline-to-rule target changes across four references: 0.

## Scope and evidence

The compressed raw archive contains 40 complete FP32 tensors, without quantization. The trace records preprocessing, native candidate rank, native suppression witnesses, native returned indices, and postclip lineage. Native outputs are reconstructed exactly from captured tensors and independently replayed before stage results are interpreted.

- These are 20 repeatedly inspected development photographs, not an independent test set.
- Raw tensors describe only the frozen PR158 reviewed-label checkpoint and the recorded fresh CPU runtime. Other checkpoints remain outside this full-stage collection.
- The archived L4 and fresh CPU runtimes differ. Numerical records and scored outcomes are compared explicitly; prediction byte identity is not presumed.
- Equal confident scores are preserved. Native tied choices are observed and checked against the active maximum score; display ranking uses anchor index only to make tied rows legible, without changing native inference.
- A best-overlap candidate is selected using annotations for diagnosis. It may depict an adjacent physical front and is not a deployable ranking or matching rule.
- Known-corner errors are recorded only where the annotation supplies that corner; predicted endpoint availability is a separate confidence test.
- Fixed-pool support and maximum one-to-one matching are upper bounds under the existing IoU relation. They do not establish recoverable accuracy, a confidence cutoff, or an optimal suppression rule.
- First-lost stages explain this computation, not the training cause of a low score or poor box. No training, loss change, label change or optimizer update is performed.
- Original and reflected postclip outputs are separately scored to expose view support; the completed two-view method still preserves original boxes and averages only eligible associated corners.
- Collection timings are local resource observations, not a detector speed benchmark or cross-hardware comparison.
- The notebook's separately packaged corner-suppression companion uses saved R68 predictions and its own fixed protocol; it does not modify these freshly collected tensors.


# Corner agreement as a condition for additional suppression

This fixed saved-output experiment compares unchanged detections, the existing additional box-only IoU 0.60 control, and the same suppression qualified by two accepted corresponding corners within five pixels. All selection uses predictions alone. The R68 reflected-output union is excluded.

| Method | Variant | r67 misses | Extras | Duplicate subset | Pair within 5 px | Pair within 10 px |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| historical_retained | unchanged | 21 | 50 | 5 | 569/583 | 583/583 |
| historical_retained | additional_nms_060 | 25 | 36 | 0 | 569/583 | 583/583 |
| historical_retained | corner_qualified_nms_060 | 21 | 47 | 5 | 569/583 | 583/583 |
| pr157_native | unchanged | 21 | 48 | 4 | 568/583 | 583/583 |
| pr157_native | additional_nms_060 | 25 | 34 | 0 | 568/583 | 583/583 |
| pr157_native | corner_qualified_nms_060 | 21 | 44 | 4 | 568/583 | 583/583 |
| pr157_qfl | unchanged | 15 | 82 | 17 | 570/583 | 582/583 |
| pr157_qfl | additional_nms_060 | 17 | 57 | 4 | 570/583 | 582/583 |
| pr157_qfl | corner_qualified_nms_060 | 15 | 76 | 16 | 570/583 | 582/583 |
| pr158_original_labels | unchanged | 50 | 84 | 6 | 541/583 | 566/583 |
| pr158_original_labels | additional_nms_060 | 51 | 61 | 2 | 541/583 | 566/583 |
| pr158_original_labels | corner_qualified_nms_060 | 51 | 81 | 6 | 541/583 | 566/583 |
| pr158_reviewed_labels | unchanged | 18 | 50 | 13 | 572/583 | 583/583 |
| pr158_reviewed_labels | additional_nms_060 | 19 | 31 | 2 | 572/583 | 583/583 |
| pr158_reviewed_labels | corner_qualified_nms_060 | 18 | 44 | 11 | 572/583 | 583/583 |

The full matrix retains five methods and four references. Fresh matching can change which surviving prediction represents a target. The ledgers preserve removed outputs, their suppressors, target gains, losses, rematches and confidence-qualified known-corner errors. Outputs keep their original coordinates and scores.

The two thresholds were fixed before the first candidate outcome measurement. This is an exploratory two-corner rule inspired by pose comparison in RMPE, with no learned pose-distance parameters or coordinate merging. Agreement between predictions can be shared error. The same development photographs support all reference comparisons.


## 4 Verify and download the complete evidence

Recheck source and every result member. Package the lossless raw tensors,
stage diagnostics, native/final comparisons and reference records, plus the
companion results under a separate `corner-suppression/` prefix. Input and
runtime identities remain explicit. No model or threshold is promoted.


In [4]:
verify_source()
RESULT = audit.package_results(OUTPUT, SOURCE_COMMIT, COMPANION)
companion_summary = json.loads((COMPANION / "summary.json").read_bytes())
manifest = {"source_commit": SOURCE_COMMIT, "notebook_code_cells": 4,
            "fresh_inference": True, "training": False, "optimizer_updates": 0,
            "labels_changed": False, "threshold_search": False, "automatic_promotion": False,
            "raw_diagnostic_matrix_rows": len(summary["matrix"]),
            "companion_matrix_rows": len(companion_summary["matrix"]),
            "companion_archived_continuity_verified": True,
            "result": {"filename": RESULT.name, "bytes": RESULT.stat().st_size,
                       "sha256": hashlib.sha256(RESULT.read_bytes()).hexdigest()}}
(WORK / "package-manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest, indent=2))
from IPython.display import FileLink
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(RESULT)))
else:
    files.download(str(RESULT))


{
  "source_commit": "41d0e6937c695377373cc55071ffb780345870ee",
  "notebook_code_cells": 4,
  "fresh_inference": true,
  "training": false,
  "optimizer_updates": 0,
  "labels_changed": false,
  "threshold_search": false,
  "automatic_promotion": false,
  "raw_diagnostic_matrix_rows": 40,
  "companion_matrix_rows": 60,
  "companion_archived_continuity_verified": true,
  "result": {
    "filename": "baomai-current-proposals-r69-results.zip",
    "bytes": 39331256,
    "sha256": "cb6a80abb90d792cb241beb93c13312f3bdcbab9c5631c831523df8cce96ca48"
  }
}


/workspace/scratch/be67d3adbd23/r69-notebook-run/work/current-proposals/baomai-current-proposals-r69-results.zip